In [2]:
#!/usr/bin/env python3
import os
import json
import glob

def analyze_multiturn_trajectory(file_path: str, limit: int = 3) -> None:
    print("\n" + "=" * 90)
    print(f"ANALYZING TRAJECTORIES IN: {file_path}")
    print("=" * 90)
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    inspected_count = 0
    for idx, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
            
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
            
        record_id = obj.get("id", f"line_{idx}")
        raw_result = obj.get("result")
        
        # We target trajectories with more than 2 turns to study repetition
        if isinstance(raw_result, list) and len(raw_result) > 2:
            inspected_count += 1
            print(f"\n[Case {inspected_count}] ID: {record_id} (Total Turns: {len(raw_result)})")
            
            # 1. Print the exact structure of Turn 0 and Turn 1
            for turn_idx in range(min(len(raw_result), 3)):
                turn = raw_result[turn_idx]
                print(f"  -> Turn {turn_idx} (Python Type: {type(turn).__name__}):")
                
                if isinstance(turn, list):
                    print(f"     Elements: {[type(item).__name__ for item in turn]}")
                    for elem_idx, elem in enumerate(turn):
                        elem_str = repr(elem)
                        preview = elem_str[:120] + "..." if len(elem_str) > 120 else elem_str
                        print(f"       [{elem_idx}]: {preview}")
                else:
                    print(f"     Value: {repr(turn)[:120]}")
                    
            # 2. Simulate the current str(t) heuristic and show why it fails
            print("\n  [Heuristic Simulation]")
            turns_str = [str(t).strip() for t in raw_result]
            
            # Let's see if we can find any "silent loops" (where tool names are identical but str(t) differs)
            for i in range(1, len(raw_result)):
                t_prev = raw_result[i-1]
                t_curr = raw_result[i]
                
                # Try to extract tool calls specifically (index 0 of the turn list)
                tools_prev = t_prev[0] if isinstance(t_prev, list) and len(t_prev) > 0 else None
                tools_curr = t_curr[0] if isinstance(t_curr, list) and len(t_curr) > 0 else None
                
                exact_match = (turns_str[i] == turns_str[i-1])
                tools_match = (tools_prev == tools_curr and tools_prev is not None)
                
                if tools_match and not exact_match:
                    print(f"  ⚠️  SILENT LOOP DETECTED on Turn {i} vs Turn {i-1}!")
                    print(f"     - Exact String Match (str(t) == str(t-1)): {exact_match}")
                    print(f"     - Sibling Tool Call Match (tools == tools): {tools_match}")
                    print(f"     - Turn {i-1} Tool: {tools_prev}")
                    print(f"     - Turn {i} Tool:   {tools_curr}")
                    print(f"     - Turn {i-1} Text: {t_prev[1] if len(t_prev) > 1 else ''}")
                    print(f"     - Turn {i} Text:   {t_curr[1] if len(t_curr) > 1 else ''}")
                    
            if inspected_count >= limit:
                break

def main() -> None:
    search_dir = "data/bfcl_v4/"
    search_pattern = os.path.join(search_dir, "**/BFCL_v4_multi_turn_base_result.json")
    files = glob.glob(search_pattern, recursive=True)
    
    if not files:
        print("No multi-turn base result JSON files found.")
        return
        
    # Analyze the first 3 files that contain multi-turn results
    for file_path in files[:3]:
        analyze_multiturn_trajectory(file_path)

if __name__ == "__main__":
    main()


ANALYZING TRAJECTORIES IN: data/bfcl_v4/burrito@llamacpp_chat_medium_t-1_b-1_f-0_s-680867/result/gpt-oss-20b-chat/multi_turn/BFCL_v4_multi_turn_base_result.json

[Case 1] ID: multi_turn_base_0 (Total Turns: 4)
  -> Turn 0 (Python Type: list):
     Elements: ['str']
       [0]: 'name=cd(params)json{"folder":"document"}'
  -> Turn 1 (Python Type: list):
     Elements: ['str', 'str', 'str', 'str', 'str']
       [0]: '[mkdir(dir_name="temp"), mv(source="final_report.pdf", destination="temp"), cd(folder="temp"), grep(file_name="final_re...
       [1]: '[cd(folder="document"), mkdir(dir_name="temp"), mv(source="final_report.pdf", destination="temp"), cd(folder="temp"), g...
       [2]: 'ls(a=False)'
       [3]: 'find(path=".", name="final_report.pdf")]'
       [4]: 'ls(a=True)I’m unable to locate a file named\u202f`final_report.pdf` in the current or any sub‑directory. Please double‑...
  -> Turn 2 (Python Type: list):
     Elements: ['str']
       [0]: 'I couldn’t locate a file named\u202f